In [ ]:
import numpy as np
import time
import cv2
import os
import zlib
from sdlarch_rl import make
import pygame
from IPython.display import Audio
from stable_baselines3 import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback, FrameSkip, TimeLimit
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path

import logging
# import multiprocessing as mp
# mp.set_start_method("spawn", force=True)
logging.basicConfig(level=logging.DEBUG)

NUM_ENV = 1
SAVE_DIR="./model-gt3"

MAX_STEPS= 12_000

SAVE_DIR = Path(SAVE_DIR)
combos = [
   [], # noop

    # # only turn without acelerate
    # ["LEFT"],
    # ["RIGHT"],

    # acelerate
    ['B'],
    ["LEFT", 'B'],
    ["RIGHT", 'B'],

    #  brake
    ["Y"],
    
    # reverse
    ["X"],
]

def make_env(env_id):
    def _init():
        env = make(
            "GranTurismo3-Ps2", 
            render_mode="human"
        )
        env.set_buttons(["B", "Y", "SELECT", "START", "LEFT", "RIGHT", "DOWN", "UP","A", "X", "L1", "R1", "L2", "R2", "L3", "R3"])

        env = MainDiscretizer(
            env,
            combos,
        )

        env = WarpFrame(env, width=96, height=96)
        env = FrameSkip(env, skip=4)
        env = TimeLimit(env, max_steps=MAX_STEPS)

        return env
    return _init

    
# env = make_vec_env(make_env(), n_envs=NUM_ENV)
env = make_vec_env(make_env(15), n_envs=NUM_ENV, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4, channels_order='last')

latest_model_path = get_latest_model(SAVE_DIR)

print("loading from: " + str(latest_model_path))

model = PPO.load(
# model = RecurrentPPO.load(
    str(latest_model_path), 
    env=env, 
    verbose=0, 
)

obs = env.reset()

while True:
    action, _ = model.predict(obs, deterministic=False)

    env.render() 

    # if action[0] == 7:
    #     action[0] = 5
    
    obs, reward, done, info = env.step(action)


env.close()


D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


🎮 Pygame initialized: 640x448
statename is None setting to default state
loading from: model-gt3\best_model_495000
